# 🗣️ Stage 2: Translation Quality Review
Side-by-side comparison of Japanese original vs Hindi translation.
Use this notebook to manually review and flag bad translations.

In [ ]:
import json
import pandas as pd
from IPython.display import display

with open('../data/processed/transcript_hi.json', 'r', encoding='utf-8') as f:
    segments = json.load(f)

df = pd.DataFrame([{
    'start': s['start'],
    'end': s['end'],
    'duration': round(s['end'] - s['start'], 2),
    'japanese': s.get('text_original', s.get('text', '')),
    'hindi': s.get('text_translated', ''),
} for s in segments])

pd.set_option('display.max_colwidth', 80)
print(f'Total segments: {len(df)}')
print(f'Total duration: {df["duration"].sum():.1f}s')
display(df.head(20))

In [ ]:
# Flag segments where translation might be missing or empty
empty = df[df['hindi'].str.strip() == '']
print(f'Empty translations: {len(empty)}')
display(empty)

In [ ]:
# Duration mismatch analysis — long segments may be hard to fit
df['char_count_hi'] = df['hindi'].str.len()
df['chars_per_sec'] = df['char_count_hi'] / df['duration']

# High chars/sec = may sound rushed when spoken
fast_segments = df[df['chars_per_sec'] > 15].sort_values('chars_per_sec', ascending=False)
print(f'Potentially fast segments (>15 chars/sec): {len(fast_segments)}')
display(fast_segments[['start', 'end', 'duration', 'hindi', 'chars_per_sec']].head(10))